[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ELTE-DSED/Intro-Data-Security/blob/main/modules/module_01_foundations/Lab1_DNN_Training_and_Robust_Models.ipynb)

# **Lab 1: DNN Training & Robust Model Baselines**

**Master's Course:** Introduction to Data Security Practicum  
**Institution:** Eötvös Loránd University (ELTE), Faculty of Informatics (IK)  
**Module 1: Foundations**

## **Learning Objectives**

By the end of this lab, you will be able to:
- **Construct & Train** a convolutional neural network (CNN) from scratch on the MNIST benchmark using PyTorch.
- **Evaluate** classification performance, loss convergence, and clean test accuracy.
- **Standardize & Wrap** PyTorch models across major security frameworks (**SecML-Torch**, **Foolbox**, and **Adversarial Robustness Toolbox - ART**), mastering how input domain bounds `(0, 1)` and gradient hooks enable adversarial auditing.
- **Load & Benchmark** standard pre-trained architectures from Torchvision alongside empirically verified robust checkpoints from **RobustBench**.
- **Investigate** model confidence calibration and the empirical trade-off between clean test accuracy and adversarial robustness.

## **Table of Contents**

1. [Setup & Imports](#setup)
2. [Dataset Loading & Preprocessing](#part1)
3. [Training a Standard DNN](#part2)
4. [Evaluating Model Performance & Security Wrappers](#part3)
   - 4.1 SecML-Torch Evaluation
   - 4.2 Model Persistence & Checkpointing
   - 4.3 Standard Security Wrappers: Foolbox & ART
5. [Loading Pre-trained & Robust Models](#part4)
   - 5.1 Preprocessing Sample Images
   - 5.2 Vision Transformer Baseline (Torchvision)
   - 5.3 Robust Models from RobustBench
   - 5.4 Comparative Prediction & Confidence Analysis
6. [Hands-on Exploratory Exercises](#part5)
7. [Conclusion & Next Steps](#conclusion)

## **Setup & Imports** <a name="setup"></a>

First, we'll install necessary libraries and import required modules.

In [ ]:
# Install required packages for standard ML, SecML-Torch, Foolbox, and ART
%pip install torch torchvision matplotlib seaborn numpy scikit-learn tqdm secml-torch foolbox adversarial-robustness-toolbox -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## **1. Dataset Loading & Preprocessing** <a name="part1"></a>

We'll work with **MNIST** (handwritten digits) as our primary dataset. MNIST is a standard benchmark for:
- Image classification
- Neural network training
- Adversarial robustness research

**Dataset Details:**
- **Training samples:** 60,000
- **Test samples:** 10,000
- **Image size:** 28×28 grayscale
- **Classes:** 10 (digits 0-9)

We import the training and testing dataset of MNIST from `torchvision`, and provide them to the dedicated data loaders.

In [ ]:
# Data preprocessing transformations
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert to tensor and scale to [0, 1]
    transforms.Normalize((0.1307,), (0.3081,))  # Normalize with MNIST mean and std
])

# Load MNIST dataset
train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    transform=transform,
    download=True
)

test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    transform=transform,
    download=True
)

# Create data loaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Batch size: {batch_size}")
print(f"Number of batches (train): {len(train_loader)}")

### **Visualize Sample Images**

In [ ]:
# Visualize some training samples
def show_images(dataset, num_samples=10):
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    axes = axes.ravel()
    
    for i in range(num_samples):
        image, label = dataset[i]
        # Denormalize for visualization
        image = image * 0.3081 + 0.1307
        axes[i].imshow(image.squeeze(), cmap='gray')
        axes[i].set_title(f'Label: {label}')
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

show_images(train_dataset)

We will train a classifier for the MNIST dataset. First, we define the model as a `torch.nn.Module`, as usually done in the `torch` library. The model is a simple fully-connected network with three layers. 

## **2. Training a Standard DNN** <a name="part2"></a>

We'll implement a simple **fully connected network (MLP)** for MNIST classification.

In [ ]:
class MNISTNet(torch.nn.Module):
    """Simple fully connected network for MNIST classification."""

    def __init__(self):
        super().__init__()
        self.fc1 = torch.nn.Linear(784, 200)
        self.fc2 = torch.nn.Linear(200, 200)
        self.fc3 = torch.nn.Linear(200, 10)

    def forward(self, x):
        x = x.flatten(1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# Initialize model
net = MNISTNet().to(device)
print(net)

### **Train the Model**

we initialize the optimizer to use for training the model. We’re using Adam with a learning rate of 1e-3, which is a good default choice for our neural network and then we will start using the `SecML-Torch` functionalities to train the previously-defined model on the MNIST dataset.

In [ ]:
# Ensure gradients are enabled (just in case they were disabled by a previous cell)
torch.set_grad_enabled(True)

# Define optimizer
optimizer = optim.Adam(net.parameters(), lr=0.001, weight_decay=1e-4)

We will use the class `secmlt.models.pytorch.base_pytorch_trainer.BasePyTorchTrainer` to prepare a training loop. This class implements the regular training loop which performs optimization steps (with the optimizer of choice) on a for loop on the batches of samples, for a given amount of epochs (passed as an input parameter). The trainer handles the forward pass, loss computation, backward pass, and parameter updates automatically.

We wrap the model into a `secmlt.models.pytorch.base_pytorch_nn.BasePyTorchClassifier` class, which provides the APIs to use models subclassing the `torch.nn.Module` within SecML-Torch. This wrapper doesn’t modify your model but adds methods that integrate with SecML-Torch’s ecosystem for attacks and defenses. Then, we can train our model by calling `model.train(dataloader=training_loader)`. This single line replaces the typical PyTorch training loop boilerplate.

In [ ]:
from secmlt.models.pytorch.base_pytorch_nn import BasePyTorchClassifier
from secmlt.models.pytorch.base_pytorch_trainer import BasePyTorchTrainer


# Training MNIST model
trainer = BasePyTorchTrainer(optimizer=optimizer, epochs=1)
model = BasePyTorchClassifier(model=net, trainer=trainer)

model.train(dataloader=train_loader)

## **3. Evaluating Model Performance & Security Wrappers** <a name="part3"></a>

### **3.1 Evaluation with SecML-Torch**
We can check how the model performs on the testing dataset by using the `secmlt.metrics.classification.Accuracy` wrapper. This provides the accuracy scoring loop that queries the model with all batches and counts how many predictions are correct. The metric automatically handles device placement and batch aggregation, returning a single accuracy value for the entire test set.

In [ ]:
from secmlt.metrics.classification import Accuracy

accuracy = Accuracy()(model, test_loader)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

### **3.2 Model Persistence & Checkpointing**

Finally, we can save our model weights using standard PyTorch persistence functionalities. To access the underlying `torch.nn.Module` from a SecML-Torch wrapper, we access the `.model` attribute.

In [ ]:
from pathlib import Path

model_path = Path("models/mnist")
if not model_path.exists():
    model_path.mkdir(parents=True, exist_ok=True)
torch.save(model.model.state_dict(), model_path / "mnist_model.pt")

After saving the model we can load it for further evaluations by simply using the `torch.load` method and wrap it again with `secmlt.models.pytorch.base_pytorch_nn.BasePyTorchClassifier`

In [ ]:
trained_net = MNISTNet()
model_weights_path = model_path / "mnist_model.pt"
model_weights = torch.load(model_weights_path, map_location="cpu")
trained_net.eval()
trained_net.load_state_dict(model_weights)
trained_model = BasePyTorchClassifier(model=trained_net, trainer=trainer)

### **3.3 Standard Security Tool Wrappers: Foolbox & ART**

When preparing a neural network for security evaluations (such as adversarial evasion attacks, poisoning, or privacy audits), raw PyTorch `nn.Module` objects cannot be passed directly into specialized attack algorithms without standardization. Security toolkits need specialized wrappers for three key reasons:

1. **Input Bounds Enforcement ($[0, 1]$ or $[0, 255]$):**
   Adversarial attacks (like FGSM, PGD, or Carlini-Wagner) iteratively optimize an additive perturbation $\delta$. Without explicit bounds, gradient steps would push pixel values into non-physical ranges (e.g., negative RGB intensities or values $> 1.0$). Security wrappers enforce valid input domains at every optimization step.

2. **Input Gradient Tracking ($\nabla_x \mathcal{L}$):**
   Standard deep learning training calculates loss gradients with respect to model parameters ($\nabla_\theta \mathcal{L}$). Adversarial attacks require gradients with respect to the **input tensor** ($\nabla_x \mathcal{L}(f(x), y)$). Toolkits like Foolbox and ART standardize input gradient computation without requiring manual `x.requires_grad_()` handling.

3. **Cross-Framework Standardization:**
   Foolbox and ART provide unified APIs across PyTorch, TensorFlow, JAX, and scikit-learn, allowing you to run the exact same attack or audit on any backend.

In [ ]:
import foolbox as fb

# Wrap the trained PyTorch network with Foolbox
# Bounds define the valid domain for image pixels (here [0.0, 1.0])
fmodel = fb.PyTorchModel(trained_net, bounds=(0.0, 1.0), device=device)

# Grab a batch of test samples to verify the wrapper
sample_images, sample_labels = next(iter(test_loader))
sample_images = sample_images.to(device)
sample_labels = sample_labels.to(device)

# Compute clean accuracy via Foolbox built-in evaluation utility
clean_acc = fb.utils.accuracy(fmodel, sample_images, sample_labels)
print(f"Foolbox Evaluated Batch Accuracy: {clean_acc * 100:.2f}%")

In [ ]:
from art.estimators.classification import PyTorchClassifier

# Wrap the trained PyTorch network with Adversarial Robustness Toolbox (ART)
# ART requires loss, input shape, number of classes, and valid clip values
criterion = nn.CrossEntropyLoss()
art_classifier = PyTorchClassifier(
    model=trained_net,
    loss=criterion,
    optimizer=optimizer,
    input_shape=(1, 28, 28),
    nb_classes=10,
    clip_values=(0.0, 1.0),
    device_type="cuda" if torch.cuda.is_available() else "cpu"
)

# ART estimators accept and return NumPy arrays
art_preds = np.argmax(art_classifier.predict(sample_images.cpu().numpy()), axis=1)
art_acc = np.mean(art_preds == sample_labels.cpu().numpy())
print(f"ART Evaluated Batch Accuracy: {art_acc * 100:.2f}%")

## **4. Loading Pre-trained & Robust Models** <a name="part4"></a>

we will use the basic functionalities of SecML-Torch to import a pre-trained model from torchvision or a robust model from RobustBench. This demonstrates how SecML-Torch can wrap any PyTorch model, whether it's a standard pre-trained model or one specifically trained for adversarial robustness.

### **4.1 Loading and Preprocessing an Image**

We'll download a sample image and prepare it for inference using the standard ImageNet preprocessing pipeline: resize to 256, center-crop to 224, and convert to tensor.

In [ ]:
import io
import json
import requests
from PIL import Image
from torchvision import transforms
from torchvision.models import get_model
from secmlt.models.pytorch.base_pytorch_nn import BasePyTorchClassifier

imagenet_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

img_url = (
    "https://raw.githubusercontent.com/ajschumacher/imagen/master/imagen/"
    "n02342885_10908_hamster.jpg"
 )
labels_url = (
    "https://raw.githubusercontent.com/"
    "anishathalye/imagenet-simple-labels/master/"
    "imagenet-simple-labels.json"
 )

headers = {"User-Agent": "Mozilla/5.0"}

# Download hamster image with timeout
resp = requests.get(img_url, headers=headers, timeout=30)
resp.raise_for_status()

# Download ImageNet labels with timeout
labels_resp = requests.get(labels_url, headers=headers, timeout=30)
labels_resp.raise_for_status()

imagenet_labels = json.loads(labels_resp.text)
img = Image.open(io.BytesIO(resp.content)).convert("RGB")

plt.imshow(img)
plt.axis('off')
plt.title("Sample ImageNet Image")
plt.show()

imagenet_input_tensor = imagenet_transform(img)

### **4.2 Importing a Pre-trained Model from Torchvision**

Now we’ll load a pre-trained Vision Transformer (ViT) model from torchvision. The model has been trained on ImageNet-1K and expects images preprocessed as shown above. We wrap it with BasePyTorchClassifier to integrate it with SecML-Torch’s functionality.

In [ ]:
imagenet_net = get_model("vit_b_16", weights="IMAGENET1K_V1")
imagenet_net.to(device)
imagenet_net.eval()

imagenet_model = BasePyTorchClassifier(imagenet_net)
print("Model loaded successfully")

### **4.3 Making Predictions with the Pre-trained Model**

Let’s use the wrapped model to classify our image. The `predict` method handles the forward pass and returns the predicted class index. We’ll map this to the human-readable ImageNet label.

In [ ]:
imagenet_pred = imagenet_model.predict(imagenet_input_tensor.unsqueeze(0).to(device))
imagenet_label = imagenet_labels[imagenet_pred.item()]
print(f"Predicted class index: {imagenet_pred.item()}")
print(f"Predicted class label: {imagenet_label}")

### **4.4 Loading a Robust Model from RobustBench**

RobustBench provides models trained for adversarial robustness. We'll load a model robust to $L_\infty$ perturbations and wrap it the same way.

Before using models from RobustBench, we need to install the RobustBench package and its dependencies.

In [ ]:
%%capture
%pip install git+https://github.com/RobustBench/robustbench.git

In [ ]:
from robustbench.utils import load_model

robust_net = load_model(model_name="Salman2020Do_R18", dataset="imagenet", threat_model="Linf")
robust_net.to(device)
robust_net.eval()

robust_model = BasePyTorchClassifier(robust_net)
print("Robust model loaded successfully")

### **4.5 Comparing Predictions**

Robust models may trade clean accuracy for resilience, but on this example both models should agree.

In [ ]:
# Compare predictions and softmax confidence distributions
robust_pred = robust_model.predict(imagenet_input_tensor.unsqueeze(0).to(device))
robust_label = imagenet_labels[robust_pred.item()]

with torch.no_grad():
    vit_logits = imagenet_net(imagenet_input_tensor.unsqueeze(0).to(device))
    vit_probs = torch.softmax(vit_logits, dim=-1)
    vit_top_prob, vit_top_idx = torch.topk(vit_probs, 3)

    robust_logits = robust_net(imagenet_input_tensor.unsqueeze(0).to(device))
    robust_probs = torch.softmax(robust_logits, dim=-1)
    robust_top_prob, robust_top_idx = torch.topk(robust_probs, 3)

print(f"Predicted class label (Robust Model): {robust_label}")
print("\n=== Standard Vision Transformer (vit_b_16) ===")
for p, idx in zip(vit_top_prob[0], vit_top_idx[0]):
    print(f"  {imagenet_labels[idx.item()]}: {p.item()*100:.2f}%")

print("\n=== Robust ResNet-18 (Salman2020Do_R18 - RobustBench) ===")
for p, idx in zip(robust_top_prob[0], robust_top_idx[0]):
    print(f"  {imagenet_labels[idx.item()]}: {p.item()*100:.2f}%")

## **5. Hands-on Exploratory Exercises** <a name="part5"></a>

Deepen your understanding of model behavior, confidence calibration, and input domain constraints before we begin crafting adversarial attacks in Lab 2.

### **Exercise 1: Prediction Confidence & Softmax Calibration Check**
**Context:** Deep neural networks are notoriously *overconfident*—they often output near 100% confidence even on borderline or ambiguous samples.
**Task:** Using the test batch, compute the maximum softmax probability for each sample. Inspect the mean confidence for correct predictions vs. incorrect predictions (or lowest-confidence samples).

In [ ]:
# --- Exercise 1 Solution & Inspection ---
with torch.no_grad():
    outputs = trained_net(sample_images)
    probabilities = torch.softmax(outputs, dim=-1)
    max_probs, preds = torch.max(probabilities, dim=-1)

correct_mask = (preds == sample_labels)
mean_conf_correct = max_probs[correct_mask].mean().item() if correct_mask.any() else 0.0
mean_conf_incorrect = max_probs[~correct_mask].mean().item() if (~correct_mask).any() else 0.0

print(f"Mean Confidence on Correct Predictions:   {mean_conf_correct * 100:.2f}%")
if (~correct_mask).any():
    print(f"Mean Confidence on Incorrect Predictions: {mean_conf_incorrect * 100:.2f}%")
else:
    min_conf, min_idx = torch.min(max_probs, dim=0)
    print(f"Lowest Confidence in batch: {min_conf.item() * 100:.2f}% (True Label: {sample_labels[min_idx].item()}, Predicted: {preds[min_idx].item()})")

### **Exercise 2: Input Bound Clipping & Validation**
**Context:** Security toolkits enforce input bounds $[0.0, 1.0]$. What happens if adversarial noise or preprocessing shifts data out of bounds?
**Task:** Create an unclipped perturbed sample and compare it against a properly clipped sample.

In [ ]:
# --- Exercise 2 Solution: Bounds Checking ---
unclipped_sample = sample_images[:1] + 0.5
print(f"Unclipped sample min: {unclipped_sample.min().item():.2f}, max: {unclipped_sample.max().item():.2f}")

# Security attacks must clamp perturbed inputs to preserve physical validity
clipped_sample = torch.clamp(unclipped_sample, 0.0, 1.0)
print(f"Clipped sample min:   {clipped_sample.min().item():.2f}, max:   {clipped_sample.max().item():.2f}")
print("✓ Clamping ensures perturbations represent valid image pixels (no sensor overflow).")

### **Exercise 3: Clean Accuracy vs. Adversarial Robustness Trade-off**
**Discussion Questions:**
1. In Section 4, we benchmarked a standard Vision Transformer (`vit_b_16`) and an adversarially trained ResNet-18 (`Salman2020Do_R18`). Why does adversarial training introduce a **"robustness tax"** (a slight drop in clean test accuracy)?
2. In what real-world security domains (e.g. autonomous vehicles, financial fraud detection, biometric authentication) is trading 2-3% clean accuracy for robust worst-case resilience an essential trade-off?

## **Conclusion & Next Steps** <a name="conclusion"></a>
---

### **What You Learned**

- **Neural Network Training:** Built and trained a DNN from scratch  
- **Model Evaluation:** Used accuracy 
- **Robust Models:** Understood the concept of adversarial robustness  
- **Model Persistence:** Saved and loaded trained models  

### **Key Takeaways**

1. **Standard models** achieve high clean accuracy but are vulnerable to adversarial attacks
2. **Robust models** trade some clean accuracy for adversarial resilience
3. **Adversarial training** is the most effective defense but computationally expensive
4. **Model architecture** affects both performance and robustness

### **Preparing for Upcoming Labs**

- **Module 2:** Evasion Attacks (FGSM, PGD & Foolbox)
- **Module 3:** Poisoning & Supply Chain Trojans (ART)
- **Module 4:** Availability & Sponge Attacks
- **Module 5:** Confidentiality & Privacy Auditing (MIA with ART)
- **Module 6:** Synthetic Data Generation (VAEs & CTGAN)
- **Module 7:** Provable Defenses (DP-SGD with Opacus & Federated Learning)
- **Module 8:** GenAI & LLM Security (Prompt Injection & Jailbreaks)

### **Additional Resources**

**Foundational Papers:**
- [Explaining and Harnessing Adversarial Examples (Goodfellow et al., 2015)](https://arxiv.org/abs/1412.6572)
- [Towards Deep Learning Models Resistant to Adversarial Attacks (Madry et al., 2018)](https://arxiv.org/abs/1706.06083)
- [Adversarial Examples Are Not Bugs, They Are Features (Ilyas et al., 2019)](https://arxiv.org/abs/1905.02175)
- [SoK: Security and Privacy in Machine Learning (Papernot et al., 2018)](https://ieeexplore.ieee.org/document/8406613)
- [The NIST Adversarial ML Framework](https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-2e2023.pdf)

**Industry Standards:**
- MITRE ATLAS: Adversarial Threat Landscape for AI Systems
- OWASP Machine Learning Security Top 10
- ISO/IEC 24029: AI Trustworthiness

**Tools & Frameworks:**
- [SecML-Torch](https://secml-torch.readthedocs.io/)
- [Microsoft Threat Modeling Tool](https://www.microsoft.com/en-us/securityengineering/sdl/threatmodeling)
- [Adversarial Robustness Toolbox (ART)](https://github.com/Trusted-AI/adversarial-robustness-toolbox)